In [18]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv,find_dotenv
# from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
# from langchain_huggingface import HuggingFacePipeline,ChatHuggingFace


In [19]:
load_dotenv(find_dotenv())

True

In [20]:
# llm = HuggingFacePipeline(
#     repo_id="google/flan-t5-base",
#     task="text-generation",
#     temperature=0.7,
#     max_new_tokens=128
# )
# model = ChatHuggingFace(model=llm)

model = ChatGroq(model_name="openai/gpt-oss-20b", temperature=0.7)





In [21]:
# create a state graph
class LLMState(TypedDict):
    question: str
    answer: str

In [22]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [23]:
# create the workflow graph
graph=StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa',llm_qa)

# add edges
graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

# compile the graph
workflow=graph.compile()

In [24]:
# execute the graph

intial_state = {'question': 'How far is moon from the earth?'}
final_state = workflow.invoke(intial_state)

print(final_state['answer'])

The Moon is on average about **384 400 km** (≈ 238 900 miles) from Earth. Its distance varies from roughly **363 300 km** at perigee to **405 500 km** at apogee.


In [25]:

model.invoke('How far is moon from the earth?').content

'The average distance from the Earth to the Moon is about **384\u202f400\u202fkm** (≈\u202f238\u202f855\u202fmi).  \n\nBecause the Moon travels in an elliptical orbit, the distance varies:\n\n| Point in orbit | Distance |\n|----------------|----------|\n| **Perigee** (closest) | ~\u202f363\u202f300\u202fkm (≈\u202f225\u202f623\u202fmi) |\n| **Apogee** (farthest) | ~\u202f405\u202f500\u202fkm (≈\u202f251\u202f968\u202fmi) |\n\nSo, depending on where the Moon is in its orbit, the distance ranges roughly from 363\u202f000\u202fkm to 406\u202f000\u202fkm.'